In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# FASE: Paso 5. Regresión (MODELOS REGULARIZADOS + IA avanzada)
# Dataset: 97 train / 42 test / 20 features
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge, Lasso, BayesianRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# FUNCIONES DE MÉTRICAS
# ==========================================
def accuracy_headcount(y_true, y_pred, tolerance=0):
    return np.mean(np.abs(y_true - y_pred) <= tolerance)

def mape(y_true, y_pred):
    mask = y_true != 0
    if mask.sum() == 0: return 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def max_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def feature_marca(feat):
    if 'Minutes_Since' in feat: return " 🕐"
    elif '_trend' in feat:      return " 📈"
    elif '_ratio' in feat:      return " 📊"
    elif 'CO2' in feat:         return " 🟢"
    elif 'Temperature' in feat: return " 🌡️"
    elif 'Humidity' in feat:    return " 💧"
    return ""

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = 'ml_normalized_grouped'
OUTPUT_DIR = 'ml_results_grouped'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLASSROOM  = '1.5'

# ==========================================
# 1. LOADING DATA
# ==========================================
print("\n" + "="*70)
print(f"AULA {CLASSROOM} - REGRESIÓN REGULARIZADA (97 train / 42 test)")
print("="*70)
print("\n1. LOADING DATA...")

X_train_norm = pd.read_excel(os.path.join(INPUT_DIR, 'X_train_normalised.xlsx')).values
X_test_norm  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test_normalised.xlsx')).values
X_train_raw  = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
X_test_raw   = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
y_train      = pd.read_excel(os.path.join(INPUT_DIR, 'y_train.xlsx')).values.ravel()
y_test       = pd.read_excel(os.path.join(INPUT_DIR, 'y_test.xlsx')).values.ravel()

STD_Y      = y_train.std()
n_samples  = len(y_train)
n_features = X_train_norm.shape[1]
ratio      = n_samples / n_features

with open(os.path.join(INPUT_DIR, 'selected_features.pkl'), 'rb') as f:
    selected_features = pickle.load(f)

print(f"   Train: {n_samples} | Test: {len(y_test)} | STD_Y: {STD_Y:.2f}")
print(f"   Features: {n_features} | Ratio: {ratio:.1f} muestras/feature")
print(f"   y_train: min={y_train.min():.0f} max={y_train.max():.0f}")
print(f"   y_test:  min={y_test.min():.0f}  max={y_test.max():.0f}")

print(f"\n   🟢 FEATURES ({n_features}):")
for i, feat in enumerate(selected_features, 1):
    print(f"   {i:2d}. {feat}{feature_marca(feat)}")

# ==========================================
# 2. DIVISIÓN INTERNA 80/20
# ==========================================
split_idx    = int(n_samples * 0.8)
X_tr80_norm  = X_train_norm[:split_idx]
X_val20_norm = X_train_norm[split_idx:]
X_tr80_raw   = X_train_raw[:split_idx]
X_val20_raw  = X_train_raw[split_idx:]
y_tr80       = y_train[:split_idx]
y_val20      = y_train[split_idx:]

print(f"\n2. VALIDACIÓN 80/20: Train={len(y_tr80)} | Val={len(y_val20)} | Test={len(y_test)}")

# ==========================================
# 3. MODELOS
# ==========================================
print("\n3. MODELOS...")

needs_scaling = {
    'Ridge (α=10)',
    'Lasso (α=1)',
    'Bayesian Ridge',
    'GP (Matern)',
    'SVR (rbf, C=0.5)',
    'SVR (linear, C=0.5)',
    'KNN (k=5)',
    'KNN (k=7)',
    'KNN (k=9)',
}

models = {
    # ── Lineales regularizados (norm) ─────────────────────
    'Ridge (α=10)':          Ridge(alpha=10.0),
    'Lasso (α=1)':           Lasso(alpha=1.0),

    # ── Bayesian Ridge (norm) ─────────────────────────────
    # Estima la regularización automáticamente — ideal para pocos datos
    'Bayesian Ridge':        BayesianRidge(),

    # ── Gaussian Process (norm) ───────────────────────────
    # Modelo probabilístico — excelente con datasets pequeños
    'GP (Matern)':           GaussianProcessRegressor(
                                 kernel=Matern(nu=1.5) + WhiteKernel(),
                                 alpha=1e-3, normalize_y=True,
                                 random_state=42),

    # ── SVR (norm) ────────────────────────────────────────
    'SVR (rbf, C=0.5)':      SVR(kernel='rbf', C=0.5, gamma='scale', epsilon=1.0),
    'SVR (linear, C=0.5)':   SVR(kernel='linear', C=0.5, epsilon=1.0),

    # ── Árboles (raw) ─────────────────────────────────────
    'Decision Tree (d=4)':   DecisionTreeRegressor(
                                 max_depth=4, min_samples_leaf=10,
                                 min_samples_split=10, random_state=42),
    'RF (depth=4, leaf=10)': RandomForestRegressor(
                                 n_estimators=100, max_depth=4,
                                 min_samples_leaf=10, min_samples_split=10,
                                 random_state=42, n_jobs=-1),
    'ExtraTrees (depth=4)':  ExtraTreesRegressor(
                                 n_estimators=100, max_depth=4,
                                 min_samples_leaf=10, min_samples_split=10,
                                 random_state=42, n_jobs=-1),
    'GB (lr=0.02, depth=2)': GradientBoostingRegressor(
                                 n_estimators=100, max_depth=2,
                                 learning_rate=0.02, min_samples_leaf=10,
                                 subsample=0.7, random_state=42),

    # ── KNN (norm) ────────────────────────────────────────
    'KNN (k=5)':             KNeighborsRegressor(n_neighbors=5, weights='distance'),
    'KNN (k=7)':             KNeighborsRegressor(n_neighbors=7, weights='distance'),
    'KNN (k=9)':             KNeighborsRegressor(n_neighbors=9, weights='distance'),
}

n_norm = sum(1 for m in models if m in needs_scaling)
n_raw  = len(models) - n_norm
print(f"   Total: {len(models)} modelos")
print(f"   📊 Normalizados: {n_norm} | 📁 Raw: {n_raw}")

# ==========================================
# 4. ENTRENAR Y EVALUAR
# ==========================================
print("\n4. TRAINING...")
print("="*70)

results        = {}
trained_models = {}
predictions    = {}

for name, model in models.items():
    scaled    = name in needs_scaling
    X_tr80    = X_tr80_norm    if scaled else X_tr80_raw
    X_val20   = X_val20_norm   if scaled else X_val20_raw
    X_tr_full = X_train_norm   if scaled else X_train_raw
    X_te      = X_test_norm    if scaled else X_test_raw

    print(f"\n   ▶ {name} ({'norm' if scaled else 'raw'})...")

    # Fase A: 80/20
    model.fit(X_tr80, y_tr80)
    y_val_pred  = np.maximum(model.predict(X_val20), 0)
    val_rmse    = np.sqrt(mean_squared_error(y_val20, y_val_pred))
    val_r2      = r2_score(y_val20, y_val_pred)
    y_tr80_pred = np.maximum(model.predict(X_tr80), 0)
    train_rmse  = np.sqrt(mean_squared_error(y_tr80, y_tr80_pred))

    # Fase B: 100% train → test
    model.fit(X_tr_full, y_train)
    y_test_pred = np.maximum(model.predict(X_te), 0)
    test_rmse   = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae    = mean_absolute_error(y_test, y_test_pred)
    test_r2     = r2_score(y_test, y_test_pred)

    acc_exacta = accuracy_headcount(y_test, y_test_pred, tolerance=0)
    acc_1      = accuracy_headcount(y_test, y_test_pred, tolerance=1)
    acc_2      = accuracy_headcount(y_test, y_test_pred, tolerance=2)
    acc_3      = accuracy_headcount(y_test, y_test_pred, tolerance=3)
    mape_val   = mape(y_test, y_test_pred)
    max_err    = max_error(y_test, y_test_pred)
    gap        = val_rmse - train_rmse

    if gap < STD_Y * 0.3:   diag = '✅ BUENO'
    elif gap < STD_Y * 0.6: diag = '🟡 REGULAR'
    else:                    diag = '🔴 OVERFIT'

    results[name] = {
        'R2_Test':    round(test_r2, 4),
        'RMSE_Test':  round(test_rmse, 2),
        'MAE_Test':   round(test_mae, 2),
        'R2_Val':     round(val_r2, 4),
        'RMSE_Val':   round(val_rmse, 2),
        'RMSE_Train': round(train_rmse, 2),
        'Gap':        round(gap, 2),
        'Acc_Exacta': round(acc_exacta, 4),
        'Acc_±1':     round(acc_1, 4),
        'Acc_±2':     round(acc_2, 4),
        'Acc_±3':     round(acc_3, 4),
        'MAPE_%':     round(mape_val, 2),
        'Max_Error':  round(max_err, 1),
        'Diagnosis':  diag,
    }
    trained_models[name] = model
    predictions[name]    = y_test_pred

    print(f"      Train: {train_rmse:.2f} | Val: {val_rmse:.2f} | Gap: {gap:.2f} → {diag}")
    print(f"      Test  → R²: {test_r2:.4f} | RMSE: {test_rmse:.2f} | Acc±2: {acc_2:.1%}")

# ==========================================
# 5. RESUMEN
# ==========================================
print("\n" + "="*70)
print("5. MODELS SUMMARY (ordenado por R² Val)")
print("="*70)

df_summary = pd.DataFrame(results).T.sort_values('R2_Val', ascending=False)
cols_show  = ['R2_Val', 'RMSE_Val', 'R2_Test', 'RMSE_Test', 'RMSE_Train',
              'Gap', 'Acc_±2', 'MAPE_%', 'Diagnosis']
print(df_summary[cols_show].to_string())

medals = ['🥇', '🥈', '🥉']
top3   = df_summary.index[:3].tolist()

print("\n" + "="*70)
print("🏆 TOP 3 MEJORES MODELOS — MÉTRICAS COMPLETAS")
print("="*70)

for rank, (medal, name) in enumerate(zip(medals, top3), 1):
    m = results[name]
    print(f"\n{medal} #{rank}: {name} ({'norm' if name in needs_scaling else 'raw'})")
    print(f"   {'─'*55}")
    print(f"   R² Val:      {m['R2_Val']:.4f}  | R² Test:    {m['R2_Test']:.4f}")
    print(f"   RMSE Val:    {m['RMSE_Val']:.2f}    | RMSE Test:  {m['RMSE_Test']:.2f}")
    print(f"   Train RMSE:  {m['RMSE_Train']:.2f}    | Gap: {m['Gap']:.2f} → {m['Diagnosis']}")
    print(f"   MAE: {m['MAE_Test']:.2f} | Max: {m['Max_Error']:.1f} | MAPE: {m['MAPE_%']:.1f}%")
    print(f"   Exacta: {m['Acc_Exacta']:.1%} | ±1: {m['Acc_±1']:.1%} | ±2: {m['Acc_±2']:.1%} | ±3: {m['Acc_±3']:.1%}")

best         = top3[0]
best_metrics = results[best]

# ==========================================
# 6. GUARDAR
# ==========================================
print("\n" + "="*70)
print("6. SAVING...")
print("="*70)

with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(trained_models[best], f)
with open(os.path.join(OUTPUT_DIR, 'all_models.pkl'), 'wb') as f:
    pickle.dump(trained_models, f)

df_summary.to_excel(os.path.join(OUTPUT_DIR, 'models_summary.xlsx'))

df_preds = pd.DataFrame({
    'Actual':    y_test,
    'Predicted': np.round(predictions[best], 1),
    'Error':     np.round(y_test - predictions[best], 1),
    'Acc_±1':    (np.abs(y_test - predictions[best]) <= 1).astype(int),
    'Acc_±2':    (np.abs(y_test - predictions[best]) <= 2).astype(int),
    'Acc_±3':    (np.abs(y_test - predictions[best]) <= 3).astype(int),
})
df_preds.to_excel(os.path.join(OUTPUT_DIR, 'predictions.xlsx'), index=False)

df_all = pd.DataFrame({'Actual': y_test})
for name, y_pred in predictions.items():
    df_all[name] = np.round(y_pred, 1)
df_all.to_excel(os.path.join(OUTPUT_DIR, 'all_predictions.xlsx'), index=False)

with open(os.path.join(OUTPUT_DIR, 'results.pkl'), 'wb') as f:
    pickle.dump(results, f)
with open(os.path.join(OUTPUT_DIR, 'predictions.pkl'), 'wb') as f:
    pickle.dump(predictions, f)

np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'),  y_test)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)

config = {
    'classroom':    CLASSROOM,
    'best_model':   best,
    'top3_models':  top3,
    'best_val_r2':  best_metrics['R2_Val'],
    'best_test_r2': best_metrics['R2_Test'],
    'best_rmse':    best_metrics['RMSE_Test'],
    'best_mae':     best_metrics['MAE_Test'],
    'strategy':     'regularized_models + GP + BayesianRidge',
    'n_train':      n_samples,
    'n_test':       len(y_test),
    'n_features':   n_features,
}
with open(os.path.join(OUTPUT_DIR, 'config.pkl'), 'wb') as f:
    pickle.dump(config, f)

print(f"\n   📁 Archivos en '{OUTPUT_DIR}/':")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")

# ==========================================
# 7. PERMUTATION IMPORTANCE — TOP 3
# ==========================================
print("\n" + "="*70)
print("7. PERMUTATION IMPORTANCE — TOP 3 MODELOS")
print("="*70)

for medal, name in zip(medals, top3):
    model     = trained_models[name]
    X_te_perm = X_test_norm if name in needs_scaling else X_test_raw
    try:
        perm_result = permutation_importance(
            model, X_te_perm, y_test,
            n_repeats=10, random_state=42,
            scoring='neg_mean_squared_error'
        )
        indices = np.argsort(perm_result.importances_mean)[::-1]
        print(f"\n   {medal} {name} (R²={results[name]['R2_Val']:.4f} | RMSE={results[name]['RMSE_Test']:.2f}):")
        print("   " + "-"*65)
        for i, idx in enumerate(indices):
            feat = selected_features[idx]
            imp  = perm_result.importances_mean[idx]
            bar  = '█' * max(0, int(imp * 50))
            print(f"   {i+1:2d}. {feat:<45s} {imp:.4f} {bar}{feature_marca(feat)}")
    except Exception as e:
        print(f"\n   {medal} {name} — ⚠️  falló: {e}")

print("\n" + "="*70)
print("✅ PASO 5 COMPLETADO!")
print("="*70)


AULA 1.5 - REGRESIÓN REGULARIZADA (97 train / 42 test)

1. LOADING DATA...
   Train: 97 | Test: 42 | STD_Y: 17.19
   Features: 20 | Ratio: 4.8 muestras/feature
   y_train: min=0 max=64
   y_test:  min=0  max=56

   🟢 FEATURES (20):
    1. Minutes_Since_Start 🕐
    2. Minutes_Since_First_Occupancy 🕐
    3. Minutes_Since_CO2_Peak 🕐
    4. Corridor_CO2_Increment_ratio_to_peak 📊
    5. Corridor_CO2_mean 🟢
    6. Corridor_CO2_min 🟢
    7. Corridor_CO2_ratio_to_peak 📊
    8. Corridor_CO2_trend 📈
    9. Corridor_Humidity_Increment_min 💧
   10. Corridor_Humidity_mean 💧
   11. Corridor_Humidity_min 💧
   12. Corridor_Humidity_std 💧
   13. Corridor_Temp_Increment_mean
   14. Corridor_Temperature_max 🌡️
   15. Corridor_Temperature_mean 🌡️
   16. Corridor_Temperature_std 🌡️
   17. Corridor_Temperature_trend 📈
   18. Outdoor_Humidity_Outdoor_Increment_min 💧
   19. Outdoor_Solar_Radiation_min
   20. Outdoor_Temperature_Outdoor_min 🌡️

2. VALIDACIÓN 80/20: Train=77 | Val=20 | Test=42

3. MODELOS...
 